# Silver data cleaning: open charge FI

In [127]:
import pandas as pd
from pathlib import Path
import openpyxl
import geopandas as gpd
import shapely
import pyogrio

print(f"openpyxl version: {openpyxl.__version__}")
print(f"gpd version: {gpd.__version__}")
print(f"shapely version: {shapely.__version__}")
print(f"pyogrio version: {pyogrio.__version__}")

print(f"Current directory: {Path.cwd()}")

openpyxl version: 3.1.5
gpd version: 1.1.4
shapely version: 2.1.2
pyogrio version: 0.13.0
Current directory: c:\DE_Course\de-week8-mini-project-team1\Silver


## Bronze -> Silver transformation

In [128]:
# Read the Bronze-level data, the CSV file into a DataFrame
bronze_df = pd.read_csv("../Bronze/open_charge_raw_FI.csv", encoding="utf-8")

# Inspect the DataFrame
print(bronze_df.shape, "\n")
# print(bronze_df.columns, "\n")
print(bronze_df.dtypes)

(4966, 86) 

UserComments                    str
PercentageSimilarity        float64
MediaItems                      str
IsRecentlyVerified             bool
DateLastVerified                str
                             ...   
AddressInfo.DistanceUnit      int64
OperatorInfo                float64
UsageType                   float64
StatusType                  float64
fetch_timestamp                 str
Length: 86, dtype: object


In [129]:
# Define the list of columns to keep and mapping, because we want to rename some columns
column_mapping = {
    "ID": "id",
    "NumberOfPoints": "number_of_points",
    "DateCreated": "date_created", # this one needs to be split later, only one key can exist in a Python dict
    "StatusType.IsOperational": "is_operational",
    "AddressInfo.Town": "town_api", # original API value, contains 1547 invalid municipalities -> not usable
    "AddressInfo.StateOrProvince": "region_api", # original API value, contains 826 invalid regions -> not usable
    "AddressInfo.Country.Title": "country",
    "AddressInfo.Latitude": "latitude",
    "AddressInfo.Longitude": "longitude",
    "fetch_timestamp": "fetch_timestamp"
}

# Select and rename
silver_df = (
    bronze_df[list(column_mapping.keys())]
    .rename(columns=column_mapping)
)

# Convert types
silver_df["date_created"] = pd.to_datetime(silver_df["date_created"])

# Derive columns
silver_df = silver_df.assign(
    year=silver_df["date_created"].dt.year,
    month=silver_df["date_created"].dt.month,
)

# Read the municipalities from the official Excel
# See https://stat.fi/en/luokitukset/tupa -> "Municipalities and Regional Divisions Based on Municipalities 2026"
municipalities_and_regions_df = pd.read_excel(
    "en26_Municipalities_and_Regional_Divisions_Based_on_Municipalities_2026_in_Finnish,_Swedish_and_English.xlsx",
    usecols="A,B,H",
    skiprows=2,
    header=None,
    names=[
        "municipality",
        "municipality_code", # needed for joining later
        "region"
    ]
)

display(municipalities_and_regions_df.head())
print(municipalities_and_regions_df.shape)
print(municipalities_and_regions_df.dtypes)

# Define the final column order explicitly
silver_df = silver_df[
    [
        "id",
        "number_of_points",
        "year",
        "month",
        "date_created",
        "is_operational",
        "town_api", # will be converted to municipality later
        "region_api", 
        "country",
        "latitude",
        "longitude",
        "fetch_timestamp",
    ]
]

# Inspect the silver DataFrame
print(silver_df.shape, "\n")
print(silver_df.columns, "\n")
print(silver_df.dtypes)

,municipality,municipality_code,region
0,Alajärvi,5,South Ostrobothnia
1,Alavieska,9,North Ostrobothnia
2,Alavus,10,South Ostrobothnia
3,Asikkala,16,Päijät-Häme
4,Askola,18,Uusimaa


(308, 3)
municipality           str
municipality_code    int64
region                 str
dtype: object
(4966, 12) 

Index(['id', 'number_of_points', 'year', 'month', 'date_created',
       'is_operational', 'town_api', 'region_api', 'country', 'latitude',
       'longitude', 'fetch_timestamp'],
      dtype='str') 

id                                int64
number_of_points                float64
year                              int32
month                             int32
date_created        datetime64[us, UTC]
is_operational                   object
town_api                            str
region_api                          str
country                             str
latitude                        float64
longitude                       float64
fetch_timestamp                     str
dtype: object


In [130]:
# Convert the data types to use the nullable Pandas dtypes (Int64, boolean, string) for Silver-layer data
# because they preserve missing values (<NA>)

silver_df["number_of_points"] = silver_df["number_of_points"].astype("Int64")

silver_df["year"] = silver_df["year"].astype("Int64")
silver_df["month"] = silver_df["month"].astype("Int64")

# print(silver_df["is_operational"].unique()) # [True, <NA>, False]
silver_df["is_operational"] = silver_df["is_operational"].astype("boolean")

# use string of Pandas instead of str
string_columns = ["town_api", "region_api", "country", "fetch_timestamp"]
silver_df[string_columns] = silver_df[string_columns].astype("string")

# use datetime for the timestamp of fetching
silver_df["fetch_timestamp"] = pd.to_datetime(silver_df["fetch_timestamp"], errors="coerce", utc=True)

# Summary of data types:
# Int64 for nullable integers
# boolean for nullable booleans
# string for text
# float64 for coordinates
# datetime for date_created and fetch_timestamp

print(silver_df.dtypes)

id                                int64
number_of_points                  Int64
year                              Int64
month                             Int64
date_created        datetime64[us, UTC]
is_operational                  boolean
town_api                         string
region_api                       string
country                          string
latitude                        float64
longitude                       float64
fetch_timestamp     datetime64[us, UTC]
dtype: object


## Use GeoPandas to find the correct municipality

### Do a spatial join with the official municipality polygons (town_api -> municipality)

In [131]:
# Source: National Land Survey of Finland
# https://www.maanmittauslaitos.fi/en/maps-and-spatial-data/datasets-and-interfaces/product-descriptions/division-administrative-areas-vector ->
# https://asiointi.maanmittauslaitos.fi/karttapaikka/tiedostopalvelu?lang=en ->
# https://asiointi.maanmittauslaitos.fi/karttapaikka/tiedostopalvelu/hallinnolliset_aluejaot_vektori?lang=en ->
# Select product: Division into administrative areas 1:10 000

# Read the GeoPackage
municipalities_gdf = gpd.read_file(
    "SuomenHallinnollisetKuntajakopohjaisetAluejaot_2026_10k.gpkg"
)

print(municipalities_gdf.shape)
display(municipalities_gdf.head())

(308, 9)


c:\DE_Course\de-week8-mini-project-team1\.venv\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'SuomenHallinnollisetKuntajakopohjaisetAluejaot_2026_10k.gpkg': 'Kunta' (default), 'Maakunta', 'Elinvoimakeskus', 'Valtakunta', 'Hyvinvointialue'. Specify layer parameter to avoid this warning.
  result = read_func(


,gml_id,natcode,namefin,nameswe,landarea,freshwarea,seawarea,totalarea,geometry
0,28026789,630,Pyhäntä,Pyhäntä,810.179993,36.570000,NaN,846.750000,"MULTIPOLYGON (((458191.217 7105060.046, 458370..."
1,28025702,309,Outokumpu,Outokumpu,446.250000,137.809998,NaN,584.059998,"MULTIPOLYGON (((617807.354 6956270.87, 617128...."
2,28020116,684,Rauma,Raumo,496.309998,13.890000,599.919983,1110.119995,"MULTIPOLYGON (((211671.977 6774130.131, 211282..."
3,28022745,005,Alajärvi,Alajärvi,1008.799988,48.029999,NaN,1056.829956,"MULTIPOLYGON (((333911.129 6968504.391, 333550..."
4,28027936,614,Posio,Posio,3039.419922,505.230011,NaN,3544.649902,"MULTIPOLYGON (((538958.119 7370322.624, 539785..."


In [132]:
# Inspect the available layers
layers = gpd.list_layers(
    "SuomenHallinnollisetKuntajakopohjaisetAluejaot_2026_10k.gpkg"
)

display(layers)

,name,geometry_type
0,Kunta,Unknown
1,Maakunta,Unknown
2,Elinvoimakeskus,Unknown
3,Valtakunta,Unknown
4,Hyvinvointialue,Unknown


In [133]:
# Read only the municipality layer
municipalities_gdf = gpd.read_file(
    "SuomenHallinnollisetKuntajakopohjaisetAluejaot_2026_10k.gpkg",
    layer="kunta"
)

# display(municipalities_gdf.head())
municipalities_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 308 entries, 0 to 307
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   gml_id      308 non-null    int64   
 1   natcode     308 non-null    str     
 2   namefin     308 non-null    str     
 3   nameswe     308 non-null    str     
 4   landarea    308 non-null    float32 
 5   freshwarea  308 non-null    float32 
 6   seawarea    77 non-null     float32 
 7   totalarea   308 non-null    float32 
 8   geometry    308 non-null    geometry
dtypes: float32(4), geometry(1), int64(1), str(3)
memory usage: 17.0 KB


In [134]:
# Inspect the columns
print(municipalities_gdf.columns.tolist())

['gml_id', 'natcode', 'namefin', 'nameswe', 'landarea', 'freshwarea', 'seawarea', 'totalarea', 'geometry']


In [135]:
# Convert charging stations to points
municipalities_gdf = municipalities_gdf[
    ["natcode", "namefin", "nameswe", "geometry"]
].rename(columns={
    "natcode": "municipality_code",
    "namefin": "municipality_fi",
    "nameswe": "municipality_sv"
})

stations_gdf = gpd.GeoDataFrame(
    silver_df,
    geometry=gpd.points_from_xy(
        silver_df["longitude"],
        silver_df["latitude"]
    ),
    crs="EPSG:4326"
)

# make sure both GeoDataFrames use the same CRS
stations_gdf = stations_gdf.to_crs(municipalities_gdf.crs)

In [136]:
# Spatial join
stations_with_municipality = gpd.sjoin(
    stations_gdf,
    municipalities_gdf,
    how="left",
    predicate="within"
)

In [137]:
# Inspect the result
stations_with_municipality[
    [
        "id",
        "town_api",
        "region_api",
        "municipality_code",
        "municipality_fi",
        "municipality_sv",
        "latitude",
        "longitude"
    ]
].tail(20)

,id,town_api,region_api,municipality_code,municipality_fi,municipality_sv,latitude,longitude
4946,24348,SaloVars,Varsinais-Suomi,734,Salo,Salo,60.418764,23.164466
4947,24347,Raisio,Varsinais-Suomi,680,Raisio,Reso,60.506355,22.159411
4948,24346,Paimio,Varsinais-Suomi,577,Paimio,Pemar,60.443490,22.605210
4949,24345,Loviisa,Uusimaa,434,Loviisa,Lovisa,60.485006,25.892513
4950,24343,Kouvola,Kymenlaakso,286,Kouvola,Kouvola,60.858750,26.567395
4951,24341,Kotka,Kymenlaakso,285,Kotka,Kotka,60.542101,26.976071
4952,24340,Imatra,Etelä-Karjala,153,Imatra,Imatra,61.186438,28.739853
4953,24339,Aura,Varsinais-Suomi,019,Aura,Aura,60.642132,22.551499
4954,24242,Ikaalinen,Pirkanmaa,143,Ikaalinen,Ikalis,61.756190,23.060389
4955,24238,Tampere,Pirkanmaa,837,Tampere,Tammerfors,61.490842,23.767333


In [138]:
# Continue...
# silver_df["municipality"] = stations_with_municipality["municipality_fi"]
# silver_df["municipality_code"] = stations_with_municipality["municipality_code"] # TODO: drop this?

# display(silver_df.head(20))

### Use a lookup table to add the official region

In [139]:
# 1. Prepare municipality polygons
# Make municipality polygon columns ready for spatial join
municipalities_lookup_gdf = (
    municipalities_gdf[
        ["municipality_code", "municipality_fi", "municipality_sv", "geometry"]
    ]
    .rename(columns={
        "municipality_code": "lookup_municipality_code",
        "municipality_fi": "lookup_municipality_fi",
        "municipality_sv": "lookup_municipality_sv",
    })
)

# 2. Create point geometries from silver_df
stations_gdf = gpd.GeoDataFrame(
    silver_df,
    geometry=gpd.points_from_xy(
        silver_df["longitude"],
        silver_df["latitude"]
    ),
    crs="EPSG:4326"
)

stations_gdf = stations_gdf.to_crs(municipalities_lookup_gdf.crs)

# 3. Spatial join: coordinates -> official municipality
stations_with_municipality = gpd.sjoin(
    stations_gdf,
    municipalities_lookup_gdf,
    how="left",
    predicate="within"
)

# Rename to the final Silver schema
silver_df = (
    pd.DataFrame(
        stations_with_municipality.drop(
            columns=["geometry", "index_right"],
            errors="ignore"
        )
    )
    .rename(columns={
        "lookup_municipality_code": "municipality_code",
        "lookup_municipality_fi": "municipality",
        "lookup_municipality_sv": "municipality_sv",
    })
)

# Add the region from the official mapping using municipality_code

# TODO DEBUG
# print(type(silver_df["municipality_code"]))
# print(silver_df.columns.tolist())
# from collections import Counter

# duplicates = [col for col, count in Counter(silver_df.columns).items() if count > 1]
# print("Duplicate columns:", duplicates)

# Make sure both municipality_code columns have the same dtype and format
silver_df["municipality_code"] = (
    silver_df["municipality_code"]
    .astype("string")
    .str.zfill(3)
)

municipalities_and_regions_df["municipality_code"] = (
    municipalities_and_regions_df["municipality_code"]
    .astype("string")
    .str.zfill(3)
)

# Keep only the lookup columns needed for the merge
region_lookup_df = (
    municipalities_and_regions_df[
        ["municipality_code", "region"]
    ]
    .drop_duplicates()
)

# Merge by municipality_code, not municipality name
silver_df = silver_df.drop(columns=["region"], errors="ignore")

silver_df = silver_df.merge(
    region_lookup_df,
    on="municipality_code",
    how="left"
)

# After this, silver_df["region"] comes from the official municipality → region mapping, not from Bronze.


In [140]:
# Verify the merge
silver_df[
    silver_df["town_api"].str.contains("Mariehamn|Maarianhamina", case=False, na=False)
][
    ["id", "town_api", "municipality", "municipality_code", "region", "latitude", "longitude"]
]

,id,town_api,municipality,municipality_code,region,latitude,longitude
2990,481650,Mariehamn,Maarianhamina,478,Åland,60.121700,19.932699
2999,481641,Mariehamn,Jomala,170,Åland,60.113701,19.914900
3003,481637,Maarianhamina,Maarianhamina,478,Åland,60.106112,19.929446
3004,481636,Maarianhamina,Maarianhamina,478,Åland,60.104419,19.942798
3006,481634,Maarianhamina,Maarianhamina,478,Åland,60.099510,19.941055
3007,481633,Mariehamn,Maarianhamina,478,Åland,60.099144,19.939142
3010,481630,Mariehamn,Maarianhamina,478,Åland,60.096777,19.941843
3013,481627,Maarianhamina,Maarianhamina,478,Åland,60.081628,19.940765
3100,307060,Mariehamn,Maarianhamina,478,Åland,60.101254,19.940489
3101,307058,Mariehamn,Maarianhamina,478,Åland,60.106796,19.942853


In [141]:
# Checks

# 1. Did the row count stay the same?
print("Silver rows:", len(silver_df))
print("Bronze rows:", len(bronze_df))

# 2. Do municipality and region exist?
print(silver_df[["municipality", "municipality_code", "region"]].dtypes)

# 3. How many rows failed municipality enrichment?
print(silver_df["municipality"].isna().sum())

# 4. How many rows failed region enrichment?
print(silver_df["region"].isna().sum())

# 5. Inspect failed rows
silver_df[silver_df["region"].isna()][
    ["id", "town_api", "municipality", "municipality_code", "region", "latitude", "longitude"]
]

Silver rows: 4966
Bronze rows: 4966
municipality            str
municipality_code    string
region                  str
dtype: object
5
5


,id,town_api,municipality,municipality_code,region,latitude,longitude
3097,308553,Kerava,NaN,<NA>,NaN,0.000000,0.000000
3686,216660,Pallo-Tysterniemi-Leiri,NaN,<NA>,NaN,81.736061,4.025088
3687,216659,Tikkurila,NaN,<NA>,NaN,78.272269,23.184454
4276,214845,Ruoholahti,NaN,<NA>,NaN,38.518174,171.632554
4277,214841,Haapalehto,NaN,<NA>,NaN,-55.811599,-134.643837


In [142]:
# Next useful checks
print("Missing municipality:", silver_df["municipality"].isna().sum())
print("Missing municipality_code:", silver_df["municipality_code"].isna().sum())
print("Missing region:", silver_df["region"].isna().sum())

Missing municipality: 5
Missing municipality_code: 5
Missing region: 5


In [143]:
# Inspect remaining failures
silver_df[silver_df["region"].isna()][
    ["id", "town_api", "municipality", "municipality_code", "region", "latitude", "longitude"]
]

,id,town_api,municipality,municipality_code,region,latitude,longitude
3097,308553,Kerava,NaN,<NA>,NaN,0.000000,0.000000
3686,216660,Pallo-Tysterniemi-Leiri,NaN,<NA>,NaN,81.736061,4.025088
3687,216659,Tikkurila,NaN,<NA>,NaN,78.272269,23.184454
4276,214845,Ruoholahti,NaN,<NA>,NaN,38.518174,171.632554
4277,214841,Haapalehto,NaN,<NA>,NaN,-55.811599,-134.643837


In [144]:
# verify that Mariehamn/Maarianhamina is now fixed
silver_df[
    silver_df["town_api"].str.contains("Mariehamn|Maarianhamina", case=False, na=False)
][
    ["id", "town_api", "municipality", "municipality_code", "region", "latitude", "longitude"]
]

,id,town_api,municipality,municipality_code,region,latitude,longitude
2990,481650,Mariehamn,Maarianhamina,478,Åland,60.121700,19.932699
2999,481641,Mariehamn,Jomala,170,Åland,60.113701,19.914900
3003,481637,Maarianhamina,Maarianhamina,478,Åland,60.106112,19.929446
3004,481636,Maarianhamina,Maarianhamina,478,Åland,60.104419,19.942798
3006,481634,Maarianhamina,Maarianhamina,478,Åland,60.099510,19.941055
3007,481633,Mariehamn,Maarianhamina,478,Åland,60.099144,19.939142
3010,481630,Mariehamn,Maarianhamina,478,Åland,60.096777,19.941843
3013,481627,Maarianhamina,Maarianhamina,478,Åland,60.081628,19.940765
3100,307060,Mariehamn,Maarianhamina,478,Åland,60.101254,19.940489
3101,307058,Mariehamn,Maarianhamina,478,Åland,60.106796,19.942853


## Silver quality checks

In [ ]:
# Quality checks

# An approximate bounding box for mainland Finland (including Åland and Lapland)
# Latitude: 59.5° to 70.5°
# Longitude: 19.0° to 32.0°
FINLAND_LAT_MIN = 59.5
FINLAND_LAT_MAX = 70.5
FINLAND_LON_MIN = 19.0
FINLAND_LON_MAX = 32.0

quality_issues = []
total_rows = len(silver_df)

# Helper function for adding failed checks
def add_quality_issue(check, details, **extra_fields):
    quality_issues.append({
        "check": check,
        "status": "FAILED",
        "details": details,
        "total_rows": total_rows,
        **extra_fields
    })


# 1. Row count check
if len(silver_df) != len(bronze_df):
    add_quality_issue(
        check="ROW_COUNT",
        details=f"Bronze rows: {len(bronze_df)}, Silver rows: {len(silver_df)}",
        bronze_rows=len(bronze_df),
        silver_rows=len(silver_df)
    )


# 2. Duplicate check by id
duplicate_ids = silver_df[silver_df.duplicated(subset=["id"], keep=False)]

if not duplicate_ids.empty:
    add_quality_issue(
        check="DUPLICATE_IDS",
        details=f"Duplicate IDs found: {duplicate_ids['id'].nunique()}",
        affected_rows=len(duplicate_ids),
        duplicate_id_count=duplicate_ids["id"].nunique()
    )


# 3. Not-null checks
required_columns = [
    "id",
    "number_of_points",
    "date_created",
    "year",
    "month",
    "is_operational",
    "town_api",
    "country",
    "latitude",
    "longitude",
    "fetch_timestamp",
]

for col in required_columns:
    missing_count = silver_df[col].isna().sum()

    if missing_count > 0:
        add_quality_issue(
            check="NOT_NULL",
            details=f"{col}: {missing_count} missing values",
            column=col,
            missing_values=missing_count,
            non_null_values=silver_df[col].count(),
            missing_pct=round(missing_count / total_rows * 100, 2)
        )


# 4. Invalid numeric values: coordinates
invalid_coordinates = silver_df[
    silver_df["latitude"].lt(FINLAND_LAT_MIN)
    | silver_df["latitude"].gt(FINLAND_LAT_MAX)
    | silver_df["longitude"].lt(FINLAND_LON_MIN)
    | silver_df["longitude"].gt(FINLAND_LON_MAX)
]

if not invalid_coordinates.empty:
    add_quality_issue(
        check="INVALID_COORDINATES",
        details=f"Coordinates outside Finland: {len(invalid_coordinates)}",
        affected_rows=len(invalid_coordinates),
        invalid_pct=round(len(invalid_coordinates) / total_rows * 100, 2)
    )


# 5. Invalid numeric values: number_of_points
invalid_number_of_points = silver_df[
    silver_df["number_of_points"].lt(0)
]

if not invalid_number_of_points.empty:
    add_quality_issue(
        check="INVALID_COORDINATES",
        details=f"Invalid coordinate rows: {len(invalid_coordinates)}",
        affected_rows=len(invalid_coordinates),
        invalid_pct=round(len(invalid_coordinates) / total_rows * 100, 2)
    )


# 6. Invalid municipality
valid_municipalities = set(
    municipalities_and_regions_df["municipality"]
    .dropna()
    .astype("string")
    .str.strip()
    .str.casefold()
)

valid_regions = set(
    municipalities_and_regions_df["region"]
    .dropna()
    .astype("string")
    .str.strip()
    .str.casefold()
)

invalid_municipalities = silver_df[
    silver_df["municipality"].notna()
    & ~silver_df["municipality"]
        .str.strip()
        .str.casefold()
        .isin(valid_municipalities)
].copy()

if not invalid_municipalities.empty:
    add_quality_issue(
        check="INVALID_MUNICIPALITY",
        details=f"Invalid municipality rows: {len(invalid_municipalities)}",
        affected_rows=len(invalid_municipalities),
        invalid_pct=round(len(invalid_municipalities) / total_rows * 100, 2)
    )


# 7. Invalid region
# Since region is derived from official municipality data, there are really only two possibilities:
# 1. The enrichment succeeded → region is guaranteed to be valid.
# 2. The enrichment failed → region is NaN.

# simply check for missing values
invalid_regions = silver_df[silver_df["region"].isna()].copy()

if not invalid_regions.empty:
    add_quality_issue(
        check="MISSING_REGION_ENRICHMENT",
        details=f"Invalid or missing region rows: {len(invalid_regions)}",
        affected_rows=len(invalid_regions),
        invalid_pct=round(len(invalid_regions) / total_rows * 100, 2)
    )


# Final quality report
quality_report = pd.DataFrame(quality_issues).fillna("") # replace NaN with blank cells so that it is more human-readable

if quality_report.empty:
    print("All quality checks passed.")
else:
    display(quality_report)

,check,status,details,total_rows,column,missing_values,non_null_values,missing_pct,affected_rows,invalid_pct
0,NOT_NULL,FAILED,number_of_points: 3066 missing values,4966,number_of_points,3066.0,1900.0,61.74,,
1,NOT_NULL,FAILED,is_operational: 1280 missing values,4966,is_operational,1280.0,3686.0,25.78,,
2,NOT_NULL,FAILED,town_api: 12 missing values,4966,town_api,12.0,4954.0,0.24,,
3,INVALID_COORDINATES,FAILED,Coordinates outside Finland: 5,4966,,,,,5.0,0.1
4,INVALID_MUNICIPALITY,FAILED,Invalid municipality rows: 12,4966,,,,,12.0,0.24
5,MISSING_REGION_ENRICHMENT,FAILED,Invalid or missing region rows: 5,4966,,,,,5.0,0.1


In [146]:
# display(silver_df.head(20))

silver_df[["town_api", "municipality", "municipality_code", "region_api", "region"]].head(20)

,town_api,municipality,municipality_code,region_api,region
0,Muonio,Muonio,498,<NA>,Lapland
1,Pudasjärvi,Pudasjärvi,615,<NA>,North Ostrobothnia
2,Oulu,Oulu,564,<NA>,North Ostrobothnia
3,Oulu,Oulu,564,<NA>,North Ostrobothnia
4,Kirkkonummi,Kirkkonummi,257,Uusimaa,Uusimaa
5,Kirkkonummi,Kirkkonummi,257,Uusimaa,Uusimaa
6,Masala,Kirkkonummi,257,<NA>,Uusimaa
7,Hämeenkyrö,Hämeenkyrö,108,Pirkanmaa,Pirkanmaa
8,Nuorgam,Utsjoki,890,<NA>,Lapland
9,Utsjoki,Utsjoki,890,<NA>,Lapland


In [147]:
# check if any municipality failed to map (If municipality is NaN, also region will bew NaN)
silver_df[silver_df["municipality"].isna()][
    ["id", "town_api", "municipality", "region_api", "region", "latitude", "longitude"]
]

,id,town_api,municipality,region_api,region,latitude,longitude
3097,308553,Kerava,NaN,<NA>,NaN,0.000000,0.000000
3686,216660,Pallo-Tysterniemi-Leiri,NaN,Etelä-Karjala,NaN,81.736061,4.025088
3687,216659,Tikkurila,NaN,Uusimaa,NaN,78.272269,23.184454
4276,214845,Ruoholahti,NaN,Uusimaa,NaN,38.518174,171.632554
4277,214841,Haapalehto,NaN,Pohjois-Pohjanmaa,NaN,-55.811599,-134.643837


In [148]:
# TODO: Drop unnecessary columns
# silver_df = silver_df.drop(columns=["date_created", "town_api"], errors="ignore")

In [149]:
# Inspect the failed data in more detail

# invalid_municipalities
# invalid_municipalities.head(20)




# failing_municipality_values = (
#     invalid_municipalities["municipality"]
#     .value_counts()
#     .sort_values(ascending=False)
# )
# display(failing_municipality_values)
# failing_municipality_values.to_csv("failing_municipality_values.csv", encoding="utf-8", index=False)



# invalid_municipalities[
#     ["town_api", "region", "latitude", "longitude"]
# ].sample(30, random_state=42)




# dump to a CSV file for debugging
# municipalities_df.to_csv("municipalities.csv", encoding="utf-8", index=False)
# invalid_municipalities.to_csv("invalid_municipalities.csv", encoding="utf-8", index=False)

# invalid_dates
# display(invalid_coordinates)